<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

## My Lane as an ML Task

My selected lane is **Refresh / Content Opportunity Scoring**.

The main task is **ranking and scoring**. Each eligible content page will receive a review-opportunity score, and pages will be ordered from highest to lowest priority.

A classification model may be used to estimate the probability that a page has a declining or review-worthy outcome. However, the final business output is not only “declining” or “not declining.” The useful output is a ranked queue that tells a content team which pages should be reviewed first.

The output may include:

- an anonymized content identifier;
- a review-opportunity score;
- an estimated decline probability;
- a suggested action;
- reason codes;
- a confidence level.

Possible actions include refresh, expand, improve CTR, protect, monitor, or take no immediate action.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

# Look for the starter dataset
data_path = Path("data/raw/content_refresh_anonymized.csv")

# In Colab, clone the public starter repo if the data is not already available
if not data_path.exists():
    repo_dir = Path("/content/flyrank-ml-internship-starter")

    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
                str(repo_dir),
            ],
            check=True,
        )

    data_path = repo_dir / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(f"Dataset loaded successfully: {len(df):,} rows")
print(f"Number of columns: {df.shape[1]}")

Dataset loaded successfully: 30,000 rows
Number of columns: 44


## 2. Target or proxy

## Target or Proxy

For the starter dataset, my provisional proxy target is:

`is_declining_proxy = 1` when `trend_direction == "down"`, otherwise `0`.

This is a rule-defined proxy based on an observed current-window measurement. It allows me to practice the ML workflow and investigate whether observable search, content, freshness, and engagement signals can help rank pages for review.

This is not an ideal final capstone target because it does not measure a future outcome. It also does not prove that refreshing a page will improve its performance.

For the full warehouse analysis, a stronger target may use separate time windows, such as:

> Features from the previous 90 days → decline during the following 30 days.

The feature window and target window must not overlap, because information from the future target period would create data leakage.

In [2]:
# Create the provisional starter proxy target
df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

target_counts = df["is_declining_proxy"].value_counts().sort_index()

print("Target values:")
print(target_counts)
print()
print(f"Declining proxy rate: {df['is_declining_proxy'].mean():.3f}")

Target values:
is_declining_proxy
0    13738
1    16262
Name: count, dtype: int64

Declining proxy rate: 0.542


## 3. Success metric

## Success Metric

My primary success metric is **Precision@50**.

Precision@50 asks:

> Of the top 50 pages ranked for review, what fraction have the positive declining proxy label?

This metric matches the real content decision because a team normally has limited review capacity. The team may only be able to inspect a small number of pages, so the quality of the top recommendations is more useful than generic accuracy across all pages.

A result will be considered useful if the learned ranking beats a transparent baseline on the same page population and the same honest validation split.

For example, a Precision@50 of 0.74 means that 37 of the top 50 recommendations have the positive proxy label.

Secondary checks may include Precision@20, Average Precision, recall, and a manual review of the top-ranked recommendations.

In [3]:
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    if len(scores) != len(labels):
        raise ValueError("Scores and labels must have the same length.")

    if k > len(labels):
        raise ValueError("K cannot be larger than the dataset.")

    ranked_indices = np.argsort(-scores)
    top_k_labels = labels[ranked_indices[:k]]

    return top_k_labels.mean()


# Transparent starter baseline: stale, visible pages ranked by impressions
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500

df["baseline_review_score"] = (
    stale.astype(int)
    * visible.astype(int)
    * df["impressions_90d"]
)

for k in (20, 50):
    result = precision_at_k(
        df["baseline_review_score"],
        df["is_declining_proxy"],
        k,
    )
    print(f"Baseline Precision@{k}: {result:.3f}")

Baseline Precision@20: 0.900
Baseline Precision@50: 0.680


## 4. The unit of analysis, as a real dataframe

## Unit of Analysis

The unit of analysis is **one anonymized content page**.

One row in the dataframe represents one page with observable search, content, freshness, and engagement signals.

For my provisional lane slice, I include pages that:

- received at least 100 impressions during the 90-day measurement window;
- are at least 90 days old.

The minimum-impression condition reduces very low-volume noise. The content-age condition removes very new pages that may not yet have enough history for a meaningful review decision.

These thresholds are provisional policy choices and may be changed after further analysis.

The anonymized identifiers are used for grouping, deduplication, and validation. Their numerical or scrambled values should not be treated as predictive features.

In [4]:
# Create the provisional lane slice
lane_df = (
    df[
        (df["impressions_90d"] >= 100)
        & (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset=["content_id"])
    .copy()
)

requested_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "engagement_rate",
    "trend_direction",
    "is_declining_proxy",
]

# Keep only columns that exist in the dataset
display_columns = [
    column for column in requested_columns
    if column in lane_df.columns
]

print(f"Full dataset rows: {len(df):,}")
print(f"Lane slice rows: {len(lane_df):,}")
print("Unit of analysis: one row = one anonymized content page")

lane_df[display_columns].head(10)

Full dataset rows: 30,000
Lane slice rows: 22,006
Unit of analysis: one row = one anonymized content page


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count,engagement_rate,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,187,20,3221.0,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,445,25,2481.0,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,141,20,3515.0,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,463,22,NaN,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,263,14,2803.0,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,147,20,3080.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,445,22,NaN,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,90,20,3807.0,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,257,104,NaN,0.00,down,1
10,content_d8ee6cc6d642,client_19581e27de,20919,324,1.55,2.2,329,104,NaN,6.75,stable,0


## 5. Why ML beats a fixed rule here
## Why ML May Beat a Fixed Rule

A fixed rule is an important and understandable baseline.

For example:

> Prioritize a page when it is at least 180 days stale and has at least 500 impressions.

However, a real refresh opportunity may depend on several interacting signals:

- impressions;
- average position;
- CTR;
- content age;
- days since the last update;
- word count;
- engagement;
- content type.

A single if-statement uses hard thresholds and may miss pages that do not meet the exact rule but still show meaningful decline or opportunity.

Machine learning may improve the ranking by combining several signals and identifying non-linear relationships. It may also assign different levels of risk instead of giving every page only a yes-or-no result.

ML is only useful if it beats the fixed rule on an honest validation split and produces recommendations that humans can understand.

If the learned model does not meaningfully improve the top-ranked queue, the simpler fixed rule may be the better operational choice.

In [5]:
# Examine where the fixed rule succeeds and fails
rule_flag = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
)

rule_summary = pd.DataFrame({
    "rule_group": np.where(
        rule_flag,
        "Stale and visible",
        "Outside fixed rule"
    ),
    "is_declining_proxy": df["is_declining_proxy"],
    "impressions_90d": df["impressions_90d"],
    "ctr": df["ctr"],
})

summary = (
    rule_summary
    .groupby("rule_group")
    .agg(
        pages=("is_declining_proxy", "size"),
        declining_rate=("is_declining_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
    )
    .round(3)
)

false_positive_count = (
    rule_flag & (df["is_declining_proxy"] == 0)
).sum()

missed_declining_count = (
    (~rule_flag) & (df["is_declining_proxy"] == 1)
).sum()

print(summary)
print()
print(
    "Pages selected by the fixed rule but not declining:",
    f"{false_positive_count:,}"
)
print(
    "Declining pages outside the fixed rule:",
    f"{missed_declining_count:,}"
)

                    pages  declining_rate  median_impressions  median_ctr
rule_group                                                               
Outside fixed rule  29983           0.542               731.0        0.07
Stale and visible      17           0.941              4429.0        0.20

Pages selected by the fixed rule but not declining: 1
Declining pages outside the fixed rule: 16,246


## Self-check

## Completed Self-Check

- [x] I selected Refresh / Content Opportunity Scoring.
- [x] I framed the main task as ranking and scoring.
- [x] I explained how classification may support the ranking.
- [x] I defined a provisional target proxy.
- [x] I explained where the proxy comes from.
- [x] I identified Precision@50 as my primary success metric.
- [x] I connected the metric to real review capacity.
- [x] I showed the unit of analysis as a real dataframe.
- [x] I stated that one row represents one anonymized content page.
- [x] I created and measured a transparent baseline.
- [x] I showed why a fixed rule can make both false-positive and false-negative recommendations.
- [x] I connected the output to real content actions.
- [x] I used careful, decision-support language.
- [x] I included no client names, URLs, domains, or private queries.
- [x] The notebook runs from top to bottom without errors.